# SETONIX: SSP R-band photometry consistency check

**2026-10-31. New diagnostic notebook; the completed three-method experiment is preserved.**

Question: does the R magnitude calculated directly from each original `EMILES_baseFe_safe` spectrum agree with the matching `BaSTI+Chabrier.dat` R magnitude once response, zero point, reference spectrum and physical scale are made explicit?

This notebook begins with **absolute magnitudes at 10 pc**. It does not use fitted weights, SSP masses or a solar R magnitude in the primary comparison. It compares all matched SSPs, not a particular galaxy or fitting range, and uses the full original spectral coverage. No SFH fit, cube photometry or `speclite` is involved.

**A numerical residual is not by itself proof of a library mismatch.** Named diagnostic conventions below are hypotheses, not certified definitions of this particular historical table. The notebook cannot declare a strict publisher-matched calibration while the exact table convention remains unverified. It never chooses the smallest residual as evidence that a convention is correct, and never rescales individual SSPs to force agreement.

## 1. Source evidence and limits of what it establishes

[Falcon-Barroso et al. (2011), section 3.2](https://arxiv.org/html/1107.2303) distinguishes photometric predictions using stellar colour relations from magnitudes synthesized from model spectra. Its revised V calibration uses a Buser--Kurucz response and a Hayes Vega reference with V=0.03; agreement is discussed at about 0.01 mag. These statements do not establish the exact R response or zero point of our particular BaSTI table.

The [MILES team's modern photometry implementation](https://github.com/miles-iac/milespy/blob/1a34f0980b79acc3a639361d9514f8b3a480110b/milespy/magnitudes.py) integrates its supplied responses without an additional wavelength factor and normalizes its Vega SED at 5556 Angstrom. The associated R/V/I response files and Vega SED are embedded below with pinned URLs and hashes. This is useful independent diagnostic provenance, not proof that the modern implementation generated the historical table.

The legacy website did not yield a verified specification during notebook construction. Accordingly the default publisher-verification flag remains false. Matching IMF/isochrones/library configuration is adopted from the project owner's provenance; exact photometric conventions are treated separately.

## 2. Configuration and running location

Run with the production `ngistenv1308` Python environment on Setonix. Run All calculates SSP photometry only. All inputs are read-only and outputs have their own fingerprinted scratch directory. The default covers the full matched SSP grid; `SSP_LIMIT` supports a small implementation smoke test whose outputs are labelled a subset.

The physical unit factor is an explicit operational assumption inherited from the previous experiment. If independent documentation specifies another scale, change it with a source citation. A correct strict comparison requires a documented response convention, Vega reference/normalization, Vega magnitude and spectrum population scale. An optional verified-profile dictionary is supported below, without inventing unavailable values.

In [ ]:
from pathlib import Path
import os
for k in ("OMP_NUM_THREADS","OPENBLAS_NUM_THREADS","MKL_NUM_THREADS"):
    os.environ[k]="1"
SSP_LIMIT = None
SSP_DIR = Path("/software/projects/pawsey1308/ngist_supplementary_public/ngistTutorial/spectralTemplates/EMILES_baseFe_safe")
ORIGINAL_SSP_DIR = SSP_DIR.parent/"EMILES_baseFe"
TABLE_FILE = Path("/home/rhuang/BaSTI+Chabrier.dat")
OUTPUT_ROOT = Path("/scratch/pawsey1308/rhuang/ssp_r_photometry_check_20261031")
PREVIOUS_RESULTS = Path("/scratch/pawsey1308/rhuang/mauve_ml_three_methods_20261031/d331b6665d204432")
NOTEBOOK_FILE = Path("/home/rhuang/20261031_SETONIX_MAUVE_SSP_R_Photometry_Check.ipynb")
SSP_TO_LSUN_PER_ANGSTROM = 1.0
LSUN_ERG_S = 3.828e33
PC_CM = 3.085677581491367e18
TEST_TOL_MAG = 1e-4
KERNEL_REPRODUCTION_TOL_MAG = 1e-11
TABLE_ROUNDING_HALF_MAG = 0.0005
RUN_WEIGHT_PROPAGATION = False

# None means no authoritative exact-table R convention has yet been certified.
# A supplied dictionary must contain: filter_path, reference_path, weighting,
# vega_mag, reference_at_5556 (None if already absolute), spectrum_unit_factor,
# lsun_erg_s, source_url and provenance_note. Files are two-column Angstrom/response
# and Angstrom/physical Vega F_lambda at Earth. weighting is "photon" or "energy".
VERIFIED_TABLE_R_PROFILE = None

## 3. Reproducible resources and input identity

Embedded numeric resources come from a pinned MILES-team repository commit; no network connection or `milespy` installation is required when running this notebook. They are written only to this experiment's output directory. Native pPXF response/solar/Vega resources and source files are hashed separately. The notebook records its source-only hash, software versions, unit assumptions and every input spectrum hash.

Where an identically named original SSP exists beside the safe library, the arrays and wavelength metadata are compared directly. File/header differences are distinguished from spectral-data differences. A missing original counterpart is reported, not interpreted as a mismatch.

In [ ]:
import ast,base64,copy,datetime,hashlib,io,json,re,sys,traceback,warnings
import numpy as np
import pandas as pd
import scipy,astropy,h5py
from astropy.io import fits
import matplotlib.pyplot as plt
from IPython.display import display
import ppxf
from ppxf import ppxf_util as util

def sha256(path):
    h=hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda:f.read(1024*1024),b""):h.update(block)
    return h.hexdigest()

def write_json(path,value):
    path=Path(path);tmp=path.with_suffix(path.suffix+".tmp")
    tmp.write_text(json.dumps(value,indent=2,default=str,allow_nan=False)+"\n");tmp.replace(path)

def identity(path):
    m=re.fullmatch(r"Ech1\.30Z([mp])(\d+\.\d+)T(\d+\.\d+)_iTp0\.00_baseFe\.fits",Path(path).name)
    if not m:raise ValueError(f"Unexpected adopted Chabrier/baseFe filename: {path}")
    return round(float(m[3]),2),round(float(m[2])*(-1 if m[1]=="m" else 1),2)

def clean_curve(wave,value,nonnegative=False):
    wave,value=np.asarray(wave,float),np.asarray(value,float)
    if wave.ndim!=1 or wave.shape!=value.shape or len(wave)<2:raise ValueError("Invalid curve shape")
    if not np.isfinite(wave).all() or not np.isfinite(value).all():raise ValueError("Nonfinite curve")
    if np.any(np.diff(wave)<=0):raise ValueError("Curve wavelengths must be unique and increasing")
    if nonnegative and (np.any(value<0) or not np.any(value>0)):raise ValueError("Invalid response")
    return wave,value

versions=dict(python=sys.version,ppxf=ppxf.__version__,numpy=np.__version__,scipy=scipy.__version__,astropy=astropy.__version__)
package=Path(util.__file__).parent
native_filters=package/"examples/FILTER.RES.txt"
native_references=package/"sps_models/spectra_sun_vega.npz"
with np.load(native_references) as ref:
    native_wave=ref["lam"].copy();native_vega=ref["flux_vega"].copy();native_sun=ref["flux_sun"].copy()
names=("IMF slope MH Age U B V R I J H K UminusV BminusV VminusR VminusI VminusJ VminusH VminusK "
       "ML_U ML_B ML_V ML_R ML_I ML_J ML_H ML_K F439W F555W F675W F814W C439_555 C555_675 C555_814").split()
tab=np.genfromtxt(TABLE_FILE,names=names,dtype=None,encoding=None,skip_header=2)
phot={}
for row in tab:
    if row["IMF"]!="Ch":continue
    k=round(float(row["Age"]),2),round(float(row["MH"]),2)
    if k in phot:raise ValueError(f"Duplicate table key {k}")
    phot[k]=row
all_files=sorted(SSP_DIR.glob("*.fits"));keys=[identity(p) for p in all_files]
assert all_files and len(set(keys))==len(keys)
assert all(k in phot for k in keys)
files=all_files if SSP_LIMIT is None else [all_files[i] for i in np.unique(np.rint(np.linspace(0,len(all_files)-1,min(SSP_LIMIT,len(all_files)))).astype(int))]
assert SSP_TO_LSUN_PER_ANGSTROM>0 and LSUN_ERG_S>0
source_only=json.loads(NOTEBOOK_FILE.read_text())
manifest=dict(versions=versions,table_hash=sha256(TABLE_FILE),native_filter_hash=sha256(native_filters),
    native_reference_hash=sha256(native_references),native_util_hash=sha256(util.__file__),
    spectrum_hashes={p.name:sha256(p) for p in all_files},n_available=len(all_files),n_selected=len(files),
    unit_factor=SSP_TO_LSUN_PER_ANGSTROM,lsun_erg_s=LSUN_ERG_S,
    verified_profile=VERIFIED_TABLE_R_PROFILE,ssp_limit=SSP_LIMIT,
    source_hash=hashlib.sha256(json.dumps([(c["cell_type"],c["source"]) for c in source_only["cells"]],sort_keys=True).encode()).hexdigest())
manifest["original_spectrum_hashes"]={p.name:sha256(ORIGINAL_SSP_DIR/p.name) for p in all_files if (ORIGINAL_SSP_DIR/p.name).is_file()}
if VERIFIED_TABLE_R_PROFILE is not None:
    manifest["verified_resource_hashes"]={str(VERIFIED_TABLE_R_PROFILE[k]):sha256(VERIFIED_TABLE_R_PROFILE[k])
                                         for k in ("filter_path","reference_path")}
run_id=hashlib.sha256(json.dumps(manifest,sort_keys=True,default=str).encode()).hexdigest()[:16]
OUT=OUTPUT_ROOT/run_id;OUT.mkdir(parents=True,exist_ok=True)
print("Output",OUT,"SSPs",len(files),"of",len(all_files),"verified exact-table R profile",VERIFIED_TABLE_R_PROFILE is not None)
print(versions)

In [ ]:
EMBEDDED_ASSETS = {'milespy/config_files/filters/Generic_Bessell.R.dat': {'url': 'https://raw.githubusercontent.com/miles-iac/milespy/1a34f0980b79acc3a639361d9514f8b3a480110b/milespy/config_files/filters/Generic_Bessell.R.dat', 'sha256': '32d605bc41174fd9c11191a2652b093550826b2f2ac55c02a3487143a9d15d48', 'text': '5500.0 0.0000000000\n5600.0 0.2300000000\n5700.0 0.7400000000\n5800.0 0.9100000000\n5900.0 0.9800000000\n6000.0 1.0000000000\n6100.0 0.9800000000\n6200.0 0.9600000000\n6300.0 0.9300000000\n6400.0 0.9000000000\n6500.0 0.8600000000\n6600.0 0.8100000000\n6700.0 0.7800000000\n6800.0 0.7200000000\n6900.0 0.6700000000\n7000.0 0.6100000000\n7100.0 0.5600000000\n7200.0 0.5100000000\n7300.0 0.4600000000\n7400.0 0.4000000000\n7500.0 0.3500000000\n8000.0 0.1400000000\n8500.0 0.0300000000\n9000.0 0.0000000000\n'}, 'milespy/config_files/filters/Generic_Bessell.V.dat': {'url': 'https://raw.githubusercontent.com/miles-iac/milespy/1a34f0980b79acc3a639361d9514f8b3a480110b/milespy/config_files/filters/Generic_Bessell.V.dat', 'sha256': '42972ea42c765558785cdb18dbb47314c1cfba6b237107516973ecb8de407571', 'text': '4700.0 0.0000000000\n4800.0 0.0300000000\n4900.0 0.1630000000\n5000.0 0.4580000000\n5100.0 0.7800000000\n5200.0 0.9670000000\n5300.0 1.0000000000\n5400.0 0.9730000000\n5500.0 0.8980000000\n5600.0 0.7920000000\n5700.0 0.6840000000\n5800.0 0.5740000000\n5900.0 0.4610000000\n6000.0 0.3590000000\n6100.0 0.2700000000\n6200.0 0.1970000000\n6300.0 0.1350000000\n6400.0 0.0810000000\n6500.0 0.0450000000\n6600.0 0.0250000000\n6700.0 0.0170000000\n6800.0 0.0130000000\n6900.0 0.0090000000\n7000.0 0.0000000000\n'}, 'milespy/config_files/filters/Generic_Bessell.I.dat': {'url': 'https://raw.githubusercontent.com/miles-iac/milespy/1a34f0980b79acc3a639361d9514f8b3a480110b/milespy/config_files/filters/Generic_Bessell.I.dat', 'sha256': '6764a41ffa0804a2b0eef6256c5fdfcd7a5c6e3d4ea4be11f11843d3ff9b45ee', 'text': '7000.0 0.0000000000\n7100.0 0.0240000000\n7200.0 0.2320000000\n7300.0 0.5550000000\n7400.0 0.7850000000\n7500.0 0.9100000000\n7600.0 0.9650000000\n7700.0 0.9850000000\n7800.0 0.9900000000\n7900.0 0.9950000000\n8000.0 1.0000000000\n8100.0 1.0000000000\n8200.0 0.9900000000\n8300.0 0.9800000000\n8400.0 0.9500000000\n8500.0 0.9100000000\n8600.0 0.8600000000\n8700.0 0.7500000000\n8800.0 0.5600000000\n8900.0 0.3300000000\n9000.0 0.1500000000\n9100.0 0.0300000000\n9200.0 0.0000000000\n'}, 'milespy/config_files/vega.sed': {'url': 'https://raw.githubusercontent.com/miles-iac/milespy/1a34f0980b79acc3a639361d9514f8b3a480110b/milespy/config_files/vega.sed', 'sha256': 'f84e36cef9c8679fcae0bd2c84cec49907d23c84e01f1afee8e38e4f9f77766d', 'text': '      91.0   0.0000E+00\n      94.0   0.0000E+00\n      96.0   0.0000E+00\n      98.0   0.0000E+00\n     100.0   0.0000E+00\n     102.0   0.0000E+00\n     104.0   0.0000E+00\n     106.0   0.0000E+00\n     108.0   0.0000E+00\n     110.0   0.0000E+00\n     114.0   0.0000E+00\n     118.0   0.0000E+00\n     121.0   0.0000E+00\n     125.0   0.0000E+00\n     127.0   0.0000E+00\n     128.0   0.0000E+00\n     131.0   0.0000E+00\n     132.0   0.0000E+00\n     134.0   0.0000E+00\n     137.0   0.0000E+00\n     140.0   0.0000E+00\n     143.0   0.0000E+00\n     147.0   0.0000E+00\n     151.0   0.0000E+00\n     155.0   0.0000E+00\n     159.0   0.0000E+00\n     162.0   0.0000E+00\n     166.0   0.0000E+00\n     170.0   0.0000E+00\n     173.0   0.0000E+00\n     177.0   0.0000E+00\n     180.0   0.0000E+00\n     182.0   0.0000E+00\n     186.0   0.0000E+00\n     191.0   0.0000E+00\n     194.0   0.0000E+00\n     198.0   0.0000E+00\n     202.0   0.0000E+00\n     205.0   0.0000E+00\n     210.0   0.0000E+00\n     216.0   0.0000E+00\n     220.0   0.0000E+00\n     223.0   0.0000E+00\n     227.0   0.0000E+00\n     230.0   0.0000E+00\n     234.0   0.0000E+00\n     240.0   0.0000E+00\n     246.0   0.0000E+00\n     252.0   0.0000E+00\n     257.0   0.0000E+00\n     260.0   0.0000E+00\n     264.0   0.0000E+00\n     269.0   0.0000E+00\n     274.0   0.0000E+00\n     279.0   0.0000E+00\n     284.0   0.0000E+00\n     290.0   0.0000E+00\n     296.0   0.0000E+00\n     301.0   0.0000E+00\n     308.0   0.0000E+00\n     318.0   0.0000E+00\n     328.0   0.0000E+00\n     338.0   0.0000E+00\n     348.0   0.0000E+00\n     357.0   0.0000E+00\n     366.0   0.0000E+00\n     375.0   0.0000E+00\n     385.0   0.0000E+00\n     395.0   0.0000E+00\n     405.0   0.0000E+00\n     414.0   0.0000E+00\n     422.0   0.0000E+00\n     430.0   0.0000E+00\n     441.0   0.0000E+00\n     451.0   0.0000E+00\n     460.0   0.0000E+00\n     470.0   0.0000E+00\n     480.0   0.0000E+00\n     490.0   0.0000E+00\n     500.0   0.0000E+00\n     506.0   7.9528E-09\n     512.0   1.3472E-08\n     520.0   2.5677E-08\n     530.0   5.5903E-08\n     540.0   1.1782E-07\n     550.0   2.4221E-07\n     560.0   4.8449E-07\n     570.0   9.4487E-07\n     580.0   1.7993E-06\n     590.0   3.3553E-06\n     600.0   6.1191E-06\n     610.0   1.0933E-05\n     620.0   1.9162E-05\n     630.0   3.2963E-05\n     640.0   5.5713E-05\n     650.0   9.2599E-05\n     658.0   1.3738E-04\n     665.0   1.9253E-04\n     675.0   3.0767E-04\n     685.0   4.8468E-04\n     695.0   7.5286E-04\n     705.0   1.1545E-03\n     716.0   1.8204E-03\n     726.0   2.7194E-03\n     735.0   3.8635E-03\n     745.0   5.6445E-03\n     755.0   8.1677E-03\n     765.0   1.1685E-02\n     775.0   1.6556E-02\n     785.0   2.3240E-02\n     795.0   3.2307E-02\n     805.0   4.4532E-02\n     815.0   6.0841E-02\n     825.0   8.2499E-02\n     835.0   1.1089E-01\n     845.0   1.4800E-01\n     855.0   1.9607E-01\n     865.0   2.5787E-01\n     875.0   3.3682E-01\n     885.0   4.3712E-01\n     895.0   5.6364E-01\n     905.0   7.2219E-01\n     915.0   9.0845E+00\n     925.0   5.5850E+01\n     935.0   9.2795E+01\n     945.0   1.5345E+02\n     955.0   2.1442E+02\n     965.0   2.8127E+02\n     975.0   2.4964E+02\n     985.0   4.0416E+02\n     995.0   4.5785E+02\n    1005.0   5.6484E+02\n    1015.0   6.4572E+02\n    1025.0   4.5655E+02\n    1035.0   8.3650E+02\n    1045.0   1.0240E+03\n    1055.0   1.1781E+03\n    1065.0   1.3099E+03\n    1075.0   1.5267E+03\n    1085.0   1.7210E+03\n    1095.0   1.8815E+03\n    1105.0   3.7418E+04\n    1115.0   1.1408E+05\n    1125.0   1.3878E+05\n    1135.0   1.5357E+05\n    1145.0   1.1278E+05\n    1155.0   8.5374E+04\n    1165.0   5.4581E+04\n    1175.0   2.1910E+04\n    1185.0   1.0826E+04\n    1195.0   6.2456E+03\n    1205.0   5.2814E+03\n    1215.0   1.4902E+03\n    1225.0   5.8215E+03\n    1235.0   1.1522E+04\n    1245.0   2.3925E+04\n    1255.0   9.8692E+04\n    1265.0   2.2744E+05\n    1275.0   8.8280E+05\n    1285.0   1.7762E+06\n    1295.0   2.4491E+06\n    1305.0   1.9417E+06\n    1315.0   3.5020E+06\n    1325.0   4.1136E+06\n    1335.0   4.0270E+06\n    1345.0   5.2616E+06\n    1355.0   5.0797E+06\n    1365.0   5.0152E+06\n    1375.0   5.6085E+06\n    1385.0   5.5310E+06\n    1395.0   6.0327E+06\n    1405.0   6.0717E+06\n    1415.0   5.7945E+06\n    1425.0   6.2361E+06\n    1435.0   5.7462E+06\n    1442.0   6.4648E+06\n    1447.0   6.9514E+06\n    1455.0   6.9318E+06\n    1465.0   6.3319E+06\n    1475.0   6.2642E+06\n    1485.0   6.5975E+06\n    1495.0   6.9629E+06\n    1505.0   7.1976E+06\n    1512.0   6.6892E+06\n    1517.0   7.0477E+06\n    1525.0   6.2649E+06\n    1535.0   5.7357E+06\n    1545.0   7.0106E+06\n    1555.0   6.4694E+06\n    1565.0   5.8913E+06\n    1575.0   6.7436E+06\n    1585.0   7.0645E+06\n    1595.0   7.5089E+06\n    1605.0   7.1014E+06\n    1615.0   6.9505E+06\n    1625.0   6.3316E+06\n    1635.0   6.7490E+06\n    1645.0   7.1037E+06\n    1655.0   5.6663E+06\n    1665.0   7.0335E+06\n    1672.0   5.0820E+06\n    1677.0   6.8618E+06\n    1685.0   7.2888E+06\n    1695.0   6.7231E+06\n    1705.0   6.5331E+06\n    1715.0   6.6172E+06\n    1725.0   6.5376E+06\n    1735.0   6.8479E+06\n    1745.0   6.6436E+06\n    1755.0   7.1074E+06\n    1765.0   6.2889E+06\n    1775.0   6.8539E+06\n    1785.0   6.7585E+06\n    1795.0   7.2091E+06\n    1805.0   6.9887E+06\n    1815.0   6.8782E+06\n    1825.0   6.6914E+06\n    1835.0   6.5421E+06\n    1845.0   6.0900E+06\n    1855.0   6.3521E+06\n    1865.0   6.4126E+06\n    1875.0   6.5354E+06\n    1885.0   6.4713E+06\n    1895.0   6.4934E+06\n    1905.0   6.2445E+06\n    1915.0   6.4565E+06\n    1925.0   6.4705E+06\n    1935.0   6.1732E+06\n    1945.0   6.5157E+06\n    1955.0   6.4476E+06\n    1967.0   6.5637E+06\n    1976.0   6.4119E+06\n    1984.0   6.3161E+06\n    1995.0   6.1585E+06\n    2005.0   6.0562E+06\n    2015.0   5.9453E+06\n    2025.0   5.8399E+06\n    2035.0   6.0028E+06\n    2045.0   6.1428E+06\n    2055.0   5.9511E+06\n    2065.0   5.8795E+06\n    2074.0   6.0495E+06\n    2078.0   6.1547E+06\n    2085.0   6.0700E+06\n    2095.0   5.9200E+06\n    2105.0   5.7597E+06\n    2115.0   5.7523E+06\n    2125.0   5.8138E+06\n    2135.0   5.2727E+06\n    2145.0   5.6329E+06\n    2155.0   5.6324E+06\n    2165.0   5.4046E+06\n    2175.0   5.3588E+06\n    2185.0   5.3984E+06\n    2195.0   5.5976E+06\n    2205.0   5.4076E+06\n    2215.0   5.1053E+06\n    2225.0   5.0637E+06\n    2235.0   5.4332E+06\n    2245.0   5.0774E+06\n    2255.0   4.9830E+06\n    2265.0   4.9712E+06\n    2275.0   5.1338E+06\n    2285.0   5.2325E+06\n    2295.0   5.0117E+06\n    2305.0   4.9875E+06\n    2315.0   4.8790E+06\n    2325.0   4.0263E+06\n    2335.0   4.6067E+06\n    2345.0   4.2922E+06\n    2355.0   4.7915E+06\n    2365.0   4.3255E+06\n    2375.0   4.5650E+06\n    2385.0   4.0666E+06\n    2395.0   4.3918E+06\n    2405.0   4.2367E+06\n    2415.0   4.3045E+06\n    2425.0   4.6045E+06\n    2435.0   4.4282E+06\n    2445.0   4.6348E+06\n    2455.0   4.4991E+06\n    2465.0   4.4967E+06\n    2475.0   4.3596E+06\n    2485.0   4.4732E+06\n    2495.0   4.3517E+06\n    2505.0   4.6820E+06\n    2513.0   4.3294E+06\n    2518.0   4.3425E+06\n    2525.0   4.1619E+06\n    2535.0   4.1016E+06\n    2545.0   4.1245E+06\n    2555.0   4.3412E+06\n    2565.0   4.3079E+06\n    2575.0   4.4829E+06\n    2585.0   4.4698E+06\n    2595.0   4.3785E+06\n    2605.0   4.2380E+06\n    2615.0   4.2144E+06\n    2625.0   4.3942E+06\n    2635.0   4.4084E+06\n    2645.0   4.7394E+06\n    2655.0   4.6995E+06\n    2665.0   4.4617E+06\n    2675.0   4.5667E+06\n    2685.0   4.5577E+06\n    2695.0   4.4166E+06\n    2705.0   4.4700E+06\n    2715.0   4.3111E+06\n    2725.0   4.3239E+06\n    2735.0   4.5048E+06\n    2745.0   4.0224E+06\n    2755.0   4.0920E+06\n    2765.0   4.2115E+06\n    2775.0   4.3603E+06\n    2785.0   4.2285E+06\n    2795.0   2.9269E+06\n    2805.0   3.2982E+06\n    2815.0   4.3734E+06\n    2825.0   4.4327E+06\n    2835.0   4.3343E+06\n    2845.0   4.2446E+06\n    2855.0   4.1635E+06\n    2865.0   4.3280E+06\n    2875.0   4.2544E+06\n    2885.0   4.2898E+06\n    2895.0   4.1994E+06\n    2910.0   4.4218E+06\n    2930.0   4.3023E+06\n    2950.0   4.2820E+06\n    2970.0   4.2857E+06\n    2990.0   4.2956E+06\n    3010.0   4.3215E+06\n    3030.0   4.2809E+06\n    3050.0   4.2508E+06\n    3070.0   4.2146E+06\n    3090.0   4.2262E+06\n    3110.0   4.1658E+06\n    3130.0   4.1464E+06\n    3150.0   4.1513E+06\n    3170.0   4.1289E+06\n    3190.0   4.0155E+06\n    3210.0   4.0645E+06\n    3230.0   4.0028E+06\n    3250.0   4.0218E+06\n    3270.0   4.0148E+06\n    3290.0   4.0133E+06\n    3310.0   4.0087E+06\n    3330.0   3.9553E+06\n    3350.0   3.9109E+06\n    3370.0   3.8989E+06\n    3390.0   3.8739E+06\n    3410.0   3.8853E+06\n    3430.0   3.8656E+06\n    3450.0   3.8436E+06\n    3470.0   3.8019E+06\n    3490.0   3.7855E+06\n    3510.0   3.7912E+06\n    3530.0   3.7796E+06\n    3550.0   3.7657E+06\n    3570.0   3.7189E+06\n    3590.0   3.6985E+06\n    3610.0   3.6784E+06\n    3630.0   3.6630E+06\n    3640.0   3.6497E+06\n    3650.0   3.6949E+06\n    3670.0   3.5524E+06\n    3690.0   3.2762E+06\n    3710.0   3.7136E+06\n    3730.0   3.9002E+06\n    3750.0   4.9651E+06\n    3770.0   5.2226E+06\n    3790.0   6.5055E+06\n    3810.0   7.7364E+06\n    3830.0   6.0617E+06\n    3850.0   8.6221E+06\n    3870.0   9.8684E+06\n    3890.0   5.1709E+06\n    3910.0   9.8028E+06\n    3930.0   9.6548E+06\n    3950.0   9.6994E+06\n    3970.0   4.9132E+06\n    3990.0   9.2423E+06\n    4010.0   1.0413E+07\n    4030.0   1.0402E+07\n    4050.0   1.0259E+07\n    4070.0   9.7331E+06\n    4090.0   7.2923E+06\n    4110.0   5.4751E+06\n    4130.0   9.1325E+06\n    4150.0   9.5756E+06\n    4170.0   9.5305E+06\n    4190.0   9.4090E+06\n    4210.0   9.3114E+06\n    4230.0   9.1331E+06\n    4250.0   9.0174E+06\n    4270.0   8.8378E+06\n    4290.0   8.7100E+06\n    4310.0   8.1903E+06\n    4330.0   5.9147E+06\n    4350.0   5.0001E+06\n    4370.0   7.8665E+06\n    4390.0   8.0983E+06\n    4410.0   8.0836E+06\n    4430.0   8.0139E+06\n    4450.0   7.9798E+06\n    4470.0   7.8756E+06\n    4490.0   7.7104E+06\n    4510.0   7.6672E+06\n    4530.0   7.5661E+06\n    4550.0   7.4215E+06\n    4570.0   7.3998E+06\n    4590.0   7.2884E+06\n    4610.0   7.2507E+06\n    4630.0   7.1043E+06\n    4650.0   7.0669E+06\n    4670.0   6.9790E+06\n    4690.0   6.8419E+06\n    4710.0   6.6515E+06\n    4730.0   6.7120E+06\n    4750.0   6.6263E+06\n    4770.0   6.5327E+06\n    4790.0   6.4312E+06\n    4810.0   6.2806E+06\n    4830.0   5.9563E+06\n    4850.0   4.5015E+06\n    4870.0   3.6215E+06\n    4890.0   5.5916E+06\n    4910.0   5.8894E+06\n    4930.0   5.8651E+06\n    4950.0   5.8570E+06\n    4970.0   5.8087E+06\n    4990.0   5.7346E+06\n    5010.0   5.6435E+06\n    5030.0   5.6034E+06\n    5050.0   5.5133E+06\n    5070.0   5.4850E+06\n    5090.0   5.4223E+06\n    5110.0   5.3593E+06\n    5130.0   5.2960E+06\n    5150.0   5.2255E+06\n    5170.0   5.1224E+06\n    5190.0   5.0752E+06\n    5210.0   5.0517E+06\n    5230.0   4.9682E+06\n    5250.0   4.9479E+06\n    5270.0   4.8553E+06\n    5290.0   4.8369E+06\n    5310.0   4.7686E+06\n    5330.0   4.7118E+06\n    5350.0   4.6819E+06\n    5370.0   4.6180E+06\n    5390.0   4.5734E+06\n    5410.0   4.5120E+06\n    5430.0   4.4707E+06\n    5450.0   4.4279E+06\n    5470.0   4.3815E+06\n    5490.0   4.3367E+06\n    5510.0   4.2905E+06\n    5530.0   4.2389E+06\n    5550.0   4.1997E+06\n    5570.0   4.1531E+06\n    5590.0   4.1100E+06\n    5610.0   4.0694E+06\n    5630.0   4.0263E+06\n    5650.0   3.9828E+06\n    5670.0   3.9389E+06\n    5690.0   3.9002E+06\n    5710.0   3.8600E+06\n    5730.0   3.8213E+06\n    5750.0   3.7793E+06\n    5770.0   3.7415E+06\n    5790.0   3.7022E+06\n    5810.0   3.6644E+06\n    5830.0   3.6269E+06\n    5850.0   3.5899E+06\n    5870.0   3.5507E+06\n    5890.0   3.4972E+06\n    5910.0   3.4813E+06\n    5930.0   3.4459E+06\n    5950.0   3.4093E+06\n    5970.0   3.3764E+06\n    5990.0   3.3397E+06\n    6010.0   3.3067E+06\n    6030.0   3.2757E+06\n    6050.0   3.2434E+06\n    6070.0   3.2123E+06\n    6090.0   3.1800E+06\n    6110.0   3.1479E+06\n    6130.0   3.1155E+06\n    6150.0   3.0699E+06\n    6170.0   3.0547E+06\n    6190.0   3.0272E+06\n    6210.0   2.9976E+06\n    6230.0   2.9676E+06\n    6250.0   2.9317E+06\n    6270.0   2.9115E+06\n    6290.0   2.8840E+06\n    6310.0   2.8537E+06\n    6330.0   2.8282E+06\n    6350.0   2.7858E+06\n    6370.0   2.7647E+06\n    6390.0   2.7474E+06\n    6410.0   2.7179E+06\n    6430.0   2.6923E+06\n    6450.0   2.6569E+06\n    6470.0   2.6340E+06\n    6490.0   2.5979E+06\n    6510.0   2.5551E+06\n    6530.0   2.4628E+06\n    6550.0   2.1068E+06\n    6570.0   1.6981E+06\n    6590.0   2.3126E+06\n    6610.0   2.4228E+06\n    6630.0   2.4321E+06\n    6650.0   2.4202E+06\n    6670.0   2.4030E+06\n    6690.0   2.3853E+06\n    6710.0   2.3664E+06\n    6730.0   2.3431E+06\n    6750.0   2.3233E+06\n    6770.0   2.3031E+06\n    6790.0   2.2824E+06\n    6810.0   2.2619E+06\n    6830.0   2.2409E+06\n    6850.0   2.2209E+06\n    6870.0   2.2016E+06\n    6890.0   2.1819E+06\n    6910.0   2.1624E+06\n    6930.0   2.1430E+06\n    6950.0   2.1239E+06\n    6970.0   2.1055E+06\n    6990.0   2.0874E+06\n    7010.0   2.0663E+06\n    7030.0   2.0497E+06\n    7050.0   2.0321E+06\n    7070.0   2.0146E+06\n    7090.0   1.9973E+06\n    7110.0   1.9713E+06\n    7130.0   1.9614E+06\n    7150.0   1.9446E+06\n    7170.0   1.9285E+06\n    7190.0   1.9120E+06\n    7210.0   1.8956E+06\n    7230.0   1.8794E+06\n    7250.0   1.8622E+06\n    7270.0   1.8480E+06\n    7290.0   1.8311E+06\n    7310.0   1.8155E+06\n    7330.0   1.8011E+06\n    7350.0   1.7858E+06\n    7370.0   1.7712E+06\n    7390.0   1.7555E+06\n    7410.0   1.7384E+06\n    7430.0   1.7237E+06\n    7450.0   1.7106E+06\n    7470.0   1.6934E+06\n    7490.0   1.6812E+06\n    7510.0   1.6691E+06\n    7530.0   1.6560E+06\n    7550.0   1.6425E+06\n    7570.0   1.6286E+06\n    7590.0   1.6153E+06\n    7610.0   1.6022E+06\n    7630.0   1.5892E+06\n    7650.0   1.5757E+06\n    7670.0   1.5624E+06\n    7690.0   1.5503E+06\n    7710.0   1.5377E+06\n    7730.0   1.5257E+06\n    7750.0   1.5129E+06\n    7770.0   1.4778E+06\n    7790.0   1.4890E+06\n    7810.0   1.4769E+06\n    7830.0   1.4645E+06\n    7850.0   1.4517E+06\n    7870.0   1.4385E+06\n    7890.0   1.4255E+06\n    7910.0   1.4173E+06\n    7930.0   1.4073E+06\n    7950.0   1.3931E+06\n    7970.0   1.3857E+06\n    7990.0   1.3670E+06\n    8010.0   1.3639E+06\n    8030.0   1.3534E+06\n    8050.0   1.3425E+06\n    8070.0   1.3322E+06\n    8090.0   1.3220E+06\n    8110.0   1.3118E+06\n    8130.0   1.3013E+06\n    8150.0   1.2913E+06\n    8170.0   1.2814E+06\n    8190.0   1.2702E+06\n    8210.0   1.2663E+06\n    8230.0   1.2575E+06\n    8250.0   1.2492E+06\n    8270.0   1.2414E+06\n    8290.0   1.2319E+06\n    8310.0   1.2212E+06\n    8330.0   1.2041E+06\n    8350.0   1.1880E+06\n    8370.0   1.1721E+06\n    8390.0   1.1490E+06\n    8410.0   1.1283E+06\n    8430.0   1.1154E+06\n    8450.0   1.1105E+06\n    8470.0   1.0673E+06\n    8490.0   1.1138E+06\n    8510.0   1.0386E+06\n    8530.0   1.1273E+06\n    8550.0   9.8014E+05\n    8570.0   1.1552E+06\n    8590.0   1.0360E+06\n    8610.0   1.0082E+06\n    8630.0   1.1758E+06\n    8650.0   1.0946E+06\n    8670.0   8.9776E+05\n    8690.0   1.1294E+06\n    8710.0   1.1867E+06\n    8730.0   1.1128E+06\n    8750.0   8.6340E+05\n    8770.0   1.0470E+06\n    8790.0   1.1625E+06\n    8810.0   1.1757E+06\n    8830.0   1.1370E+06\n    8850.0   9.8869E+05\n    8870.0   8.3220E+05\n    8890.0   1.0663E+06\n    8910.0   1.1321E+06\n    8930.0   1.1417E+06\n    8950.0   1.1363E+06\n    8970.0   1.1114E+06\n    8990.0   1.0412E+06\n    9010.0   8.1429E+05\n    9030.0   8.9231E+05\n    9050.0   1.0443E+06\n    9070.0   1.0740E+06\n    9090.0   1.0729E+06\n    9110.0   1.0783E+06\n    9130.0   1.0739E+06\n    9150.0   1.0642E+06\n    9170.0   1.0478E+06\n    9190.0   1.0145E+06\n    9210.0   9.2386E+05\n    9230.0   7.1435E+05\n    9250.0   8.8611E+05\n    9270.0   9.7090E+05\n    9290.0   1.0018E+06\n    9310.0   1.0044E+06\n    9330.0   1.0008E+06\n    9350.0   9.9551E+05\n    9370.0   9.8887E+05\n    9390.0   9.8058E+05\n    9410.0   9.6762E+05\n    9430.0   9.6621E+05\n    9450.0   9.5776E+05\n    9470.0   9.4670E+05\n    9490.0   9.3073E+05\n    9510.0   8.9732E+05\n    9530.0   8.0410E+05\n    9550.0   6.4460E+05\n    9570.0   8.1311E+05\n    9590.0   8.7883E+05\n    9610.0   8.9303E+05\n    9630.0   8.9158E+05\n    9650.0   8.9272E+05\n    9670.0   8.8422E+05\n    9690.0   8.8217E+05\n    9710.0   8.7823E+05\n    9730.0   8.7209E+05\n    9750.0   8.6599E+05\n    9770.0   8.6025E+05\n    9790.0   8.5423E+05\n    9810.0   8.4795E+05\n    9830.0   8.4202E+05\n    9850.0   8.3583E+05\n    9870.0   8.2998E+05\n    9890.0   8.2203E+05\n    9910.0   8.1658E+05\n    9930.0   8.0995E+05\n    9950.0   8.0306E+05\n    9970.0   7.9441E+05\n    9990.0   7.8192E+05\n   10025.0   6.9832E+05\n   10075.0   6.6305E+05\n   10125.0   7.4776E+05\n   10175.0   7.4477E+05\n   10225.0   7.3521E+05\n   10275.0   7.2296E+05\n   10325.0   7.1092E+05\n   10375.0   6.9934E+05\n   10425.0   6.8769E+05\n   10475.0   6.7513E+05\n   10525.0   6.6440E+05\n   10575.0   6.5411E+05\n   10625.0   6.4372E+05\n   10675.0   6.2823E+05\n   10725.0   6.2030E+05\n   10775.0   6.1146E+05\n   10825.0   5.9943E+05\n   10875.0   5.8075E+05\n   10925.0   4.8402E+05\n   10975.0   5.2915E+05\n   11025.0   5.5987E+05\n   11075.0   5.5581E+05\n   11125.0   5.4864E+05\n   11175.0   5.4038E+05\n   11225.0   5.3225E+05\n   11275.0   5.2023E+05\n   11325.0   5.1261E+05\n   11375.0   5.0765E+05\n   11425.0   4.9977E+05\n   11475.0   4.9269E+05\n   11525.0   4.8549E+05\n   11575.0   4.7795E+05\n   11625.0   4.6963E+05\n   11675.0   4.6122E+05\n   11725.0   4.5707E+05\n   11775.0   4.4671E+05\n   11825.0   4.4230E+05\n   11875.0   4.3497E+05\n   11925.0   4.3070E+05\n   11975.0   4.2335E+05\n   12025.0   4.1817E+05\n   12075.0   4.1163E+05\n   12125.0   4.0580E+05\n   12175.0   4.0025E+05\n   12225.0   3.9437E+05\n   12275.0   3.8858E+05\n   12325.0   3.8306E+05\n   12375.0   3.7782E+05\n   12425.0   3.7265E+05\n   12475.0   3.6697E+05\n   12525.0   3.6214E+05\n   12575.0   3.5623E+05\n   12625.0   3.5135E+05\n   12675.0   3.4615E+05\n   12725.0   3.3937E+05\n   12775.0   3.2275E+05\n   12825.0   2.7632E+05\n   12875.0   3.1758E+05\n   12925.0   3.2177E+05\n   12975.0   3.1876E+05\n   13025.0   3.1525E+05\n   13075.0   3.1127E+05\n   13125.0   3.0699E+05\n   13175.0   3.0242E+05\n   13225.0   2.9893E+05\n   13275.0   2.9482E+05\n   13325.0   2.9092E+05\n   13375.0   2.8707E+05\n   13425.0   2.8327E+05\n   13475.0   2.7952E+05\n   13525.0   2.7566E+05\n   13575.0   2.7217E+05\n   13625.0   2.6872E+05\n   13675.0   2.6516E+05\n   13725.0   2.6116E+05\n   13775.0   2.5832E+05\n   13825.0   2.5504E+05\n   13875.0   2.5196E+05\n   13925.0   2.4861E+05\n   13975.0   2.4545E+05\n   14025.0   2.4233E+05\n   14075.0   2.3940E+05\n   14125.0   2.3621E+05\n   14175.0   2.3335E+05\n   14225.0   2.3038E+05\n   14275.0   2.2759E+05\n   14325.0   2.2469E+05\n   14375.0   2.2197E+05\n   14425.0   2.1799E+05\n   14475.0   2.1648E+05\n   14525.0   2.1315E+05\n   14570.0   2.1141E+05\n   14620.0   2.0898E+05\n   14675.0   2.0645E+05\n   14725.0   2.0408E+05\n   14775.0   2.0133E+05\n   14825.0   1.9915E+05\n   14875.0   1.9646E+05\n   14925.0   1.9434E+05\n   14975.0   1.9157E+05\n   15025.0   1.8884E+05\n   15075.0   1.8601E+05\n   15125.0   1.8347E+05\n   15175.0   1.8057E+05\n   15225.0   1.7822E+05\n   15275.0   1.7538E+05\n   15325.0   1.7271E+05\n   15375.0   1.7210E+05\n   15425.0   1.6720E+05\n   15475.0   1.6838E+05\n   15525.0   1.6617E+05\n   15575.0   1.5930E+05\n   15625.0   1.6663E+05\n   15675.0   1.5837E+05\n   15725.0   1.5325E+05\n   15775.0   1.6227E+05\n   15825.0   1.5945E+05\n   15875.0   1.4323E+05\n   15925.0   1.5238E+05\n   15975.0   1.5777E+05\n   16050.0   1.5083E+05\n   16150.0   1.4080E+05\n   16250.0   1.4884E+05\n   16350.0   1.4018E+05\n   16450.0   1.3128E+05\n   16550.0   1.3955E+05\n   16650.0   1.3669E+05\n   16750.0   1.2801E+05\n   16850.0   1.1984E+05\n   16950.0   1.2783E+05\n   17050.0   1.2561E+05\n   17150.0   1.2292E+05\n   17250.0   1.1919E+05\n   17350.0   1.0507E+05\n   17450.0   1.1253E+05\n   17550.0   1.1262E+05\n   17650.0   1.1057E+05\n   17750.0   1.0838E+05\n   17850.0   1.0613E+05\n   17950.0   1.0393E+05\n   18050.0   1.0085E+05\n   18150.0   8.9977E+04\n   18250.0   9.4331E+04\n   18350.0   9.5443E+04\n   18450.0   9.3706E+04\n   18550.0   9.1741E+04\n   18650.0   8.9036E+04\n   18750.0   7.9245E+04\n   18850.0   8.5216E+04\n   18950.0   8.4653E+04\n   19050.0   8.3353E+04\n   19150.0   8.1733E+04\n   19250.0   7.9915E+04\n   19350.0   7.7362E+04\n   19450.0   6.8984E+04\n   19550.0   7.4446E+04\n   19650.0   7.4063E+04\n   19750.0   7.2815E+04\n   19850.0   7.1474E+04\n   19950.0   7.0142E+04\n   20050.0   6.8847E+04\n   20150.0   6.7575E+04\n   20250.0   6.6332E+04\n   20350.0   6.5117E+04\n   20450.0   6.3937E+04\n   20550.0   6.2776E+04\n   20650.0   6.1636E+04\n   20750.0   6.0535E+04\n   20850.0   5.9452E+04\n   20950.0   5.8394E+04\n   21050.0   5.7360E+04\n   21150.0   5.6303E+04\n   21250.0   5.5310E+04\n   21350.0   5.4359E+04\n   21450.0   5.3299E+04\n   21550.0   5.1811E+04\n   21650.0   4.6588E+04\n   21750.0   4.9640E+04\n   21850.0   4.9683E+04\n   21950.0   4.9013E+04\n   22050.0   4.8175E+04\n   22150.0   4.7338E+04\n   22250.0   4.6574E+04\n   22350.0   4.5792E+04\n   22450.0   4.5028E+04\n   22550.0   4.4276E+04\n   22650.0   4.3553E+04\n   22750.0   4.2835E+04\n   22850.0   4.2156E+04\n   22950.0   4.1471E+04\n   23050.0   4.0807E+04\n   23150.0   4.0153E+04\n   23250.0   3.9509E+04\n   23350.0   3.8880E+04\n   23450.0   3.8260E+04\n   23550.0   3.7639E+04\n   23650.0   3.7026E+04\n   23750.0   3.6402E+04\n   23850.0   3.5802E+04\n   23950.0   3.5195E+04\n   24050.0   3.4608E+04\n   24150.0   3.4013E+04\n   24250.0   3.3468E+04\n   24350.0   3.2936E+04\n   24450.0   3.2341E+04\n   24550.0   3.1959E+04\n   24650.0   3.1320E+04\n   24750.0   3.0886E+04\n   24850.0   3.0663E+04\n   24950.0   2.9565E+04\n   25050.0   3.0232E+04\n   25150.0   2.9476E+04\n   25250.0   2.8044E+04\n   25350.0   2.8425E+04\n   25450.0   2.8313E+04\n   25550.0   2.7559E+04\n   25650.0   2.6146E+04\n   25750.0   2.6870E+04\n   25850.0   2.6739E+04\n   25950.0   2.6302E+04\n   26050.0   2.5340E+04\n   26150.0   2.3998E+04\n   26250.0   2.2863E+04\n   26350.0   2.4344E+04\n   26450.0   2.4481E+04\n   26550.0   2.4153E+04\n   26650.0   2.3482E+04\n   26750.0   2.1853E+04\n   26850.0   2.2826E+04\n   26950.0   2.2863E+04\n   27050.0   2.2621E+04\n   27150.0   2.2332E+04\n   27250.0   2.2007E+04\n   27350.0   2.1638E+04\n   27450.0   2.1158E+04\n   27550.0   1.9611E+04\n   27650.0   2.0030E+04\n   27750.0   2.0404E+04\n   27850.0   2.0230E+04\n   27950.0   1.9990E+04\n   28050.0   1.9718E+04\n   28150.0   1.9450E+04\n   28250.0   1.9188E+04\n   28350.0   1.8934E+04\n   28450.0   1.8649E+04\n   28550.0   1.8316E+04\n   28650.0   1.7626E+04\n   28750.0   1.6488E+04\n   28850.0   1.7448E+04\n   28950.0   1.7406E+04\n   29050.0   1.7244E+04\n   29150.0   1.7013E+04\n   29250.0   1.6798E+04\n   29350.0   1.6580E+04\n   29450.0   1.6367E+04\n   29550.0   1.6153E+04\n   29650.0   1.5946E+04\n   29750.0   1.5744E+04\n   29850.0   1.5541E+04\n   29950.0   1.5344E+04\n   30050.0   1.5149E+04\n   30150.0   1.4910E+04\n   30250.0   1.4612E+04\n   30350.0   1.3601E+04\n   30450.0   1.3716E+04\n   30550.0   1.4105E+04\n   30650.0   1.4013E+04\n   30750.0   1.3868E+04\n   30850.0   1.3693E+04\n   30950.0   1.3526E+04\n   31050.0   1.3359E+04\n   31150.0   1.3196E+04\n   31250.0   1.3035E+04\n   31350.0   1.2878E+04\n   31450.0   1.2718E+04\n   31550.0   1.2568E+04\n   31650.0   1.2417E+04\n   31750.0   1.2265E+04\n   31850.0   1.2120E+04\n   31950.0   1.1973E+04\n   32100.0   1.1760E+04\n   32300.0   1.1483E+04\n   32500.0   1.1214E+04\n   32700.0   1.0920E+04\n   32900.0   1.0129E+04\n   33100.0   1.0245E+04\n   33300.0   1.0173E+04\n   33500.0   9.9561E+03\n   33700.0   9.7301E+03\n   33900.0   9.5165E+03\n   34100.0   9.3046E+03\n   34300.0   9.0996E+03\n   34500.0   8.8987E+03\n   34700.0   8.7018E+03\n   34900.0   8.5113E+03\n   35100.0   8.3221E+03\n   35300.0   8.1414E+03\n   35500.0   7.9643E+03\n   35700.0   7.7860E+03\n   35900.0   7.6297E+03\n   36100.0   7.4487E+03\n   36300.0   7.3123E+03\n   36500.0   7.1266E+03\n   36700.0   7.0247E+03\n   36900.0   6.8078E+03\n   37100.0   6.7215E+03\n   37300.0   6.4234E+03\n   37500.0   6.1888E+03\n   37700.0   6.3258E+03\n   37900.0   6.2216E+03\n   38100.0   6.0099E+03\n   38300.0   5.9023E+03\n   38500.0   5.8634E+03\n   38700.0   5.7509E+03\n   38900.0   5.5988E+03\n   39100.0   5.3730E+03\n   39300.0   5.4039E+03\n   39500.0   5.3243E+03\n   39700.0   5.2232E+03\n   39900.0   5.1202E+03\n   40100.0   4.9163E+03\n   40300.0   4.7920E+03\n   40500.0   4.6022E+03\n   40700.0   4.7091E+03\n   40900.0   4.6506E+03\n   41100.0   4.5664E+03\n   41300.0   4.4784E+03\n   41500.0   4.3657E+03\n   41700.0   4.1377E+03\n   41900.0   4.2025E+03\n   42100.0   4.1559E+03\n   42300.0   4.0848E+03\n   42500.0   4.0100E+03\n   42700.0   3.9380E+03\n   42900.0   3.8671E+03\n   43100.0   3.7974E+03\n   43300.0   3.7304E+03\n   43500.0   3.6487E+03\n   43700.0   3.4568E+03\n   43900.0   3.4752E+03\n   44100.0   3.4684E+03\n   44300.0   3.4127E+03\n   44500.0   3.3533E+03\n   44700.0   3.3099E+03\n   44900.0   3.2537E+03\n   45100.0   3.1969E+03\n   45300.0   3.1424E+03\n   45500.0   3.0888E+03\n   45700.0   3.0374E+03\n   45900.0   2.9854E+03\n   46100.0   2.9313E+03\n   46300.0   2.8655E+03\n   46500.0   2.6856E+03\n   46700.0   2.6420E+03\n   46900.0   2.7095E+03\n   47100.0   2.6947E+03\n   47300.0   2.6558E+03\n   47500.0   2.6123E+03\n   47700.0   2.5706E+03\n   47900.0   2.5283E+03\n   48100.0   2.4879E+03\n   48300.0   2.4481E+03\n   48500.0   2.4088E+03\n   48700.0   2.3701E+03\n   48900.0   2.3332E+03\n   49100.0   2.2968E+03\n   49300.0   2.2597E+03\n   49500.0   2.2244E+03\n   49700.0   2.1895E+03\n   49900.0   2.1563E+03\n   50100.0   2.1224E+03\n   50300.0   2.0902E+03\n   50500.0   2.0572E+03\n   50700.0   2.0258E+03\n   50900.0   1.9926E+03\n   51100.0   1.9426E+03\n   51300.0   1.8363E+03\n   51500.0   1.8899E+03\n   51700.0   1.8753E+03\n   51900.0   1.8498E+03\n   52100.0   1.8212E+03\n   52300.0   1.7942E+03\n   52500.0   1.7686E+03\n   52700.0   1.7422E+03\n   52900.0   1.7162E+03\n   53100.0   1.6916E+03\n   53300.0   1.6663E+03\n   53500.0   1.6423E+03\n   53700.0   1.6187E+03\n   53900.0   1.5953E+03\n   54100.0   1.5723E+03\n   54300.0   1.5496E+03\n   54500.0   1.5281E+03\n   54700.0   1.5059E+03\n   54900.0   1.4850E+03\n   55100.0   1.4634E+03\n   55300.0   1.4430E+03\n   55500.0   1.4229E+03\n   55700.0   1.4031E+03\n   55900.0   1.3834E+03\n   56100.0   1.3641E+03\n   56300.0   1.3459E+03\n   56500.0   1.3270E+03\n   56700.0   1.3092E+03\n   56900.0   1.2908E+03\n   57100.0   1.2735E+03\n   57300.0   1.2564E+03\n   57500.0   1.2386E+03\n   57700.0   1.2219E+03\n   57900.0   1.2055E+03\n   58100.0   1.1892E+03\n   58300.0   1.1740E+03\n   58500.0   1.1607E+03\n   58700.0   1.1424E+03\n   58900.0   1.1156E+03\n   59100.0   1.0583E+03\n   59300.0   1.0912E+03\n   59500.0   1.0848E+03\n   59700.0   1.0716E+03\n   59900.0   1.0578E+03\n   60100.0   1.0441E+03\n   60300.0   1.0306E+03\n   60500.0   1.0173E+03\n   60700.0   1.0041E+03\n   60900.0   9.9101E+02\n   61100.0   9.7891E+02\n   61300.0   9.6615E+02\n   61500.0   9.5433E+02\n   61700.0   9.4185E+02\n   61900.0   9.3030E+02\n   62100.0   9.1809E+02\n   62300.0   9.0680E+02\n   62500.0   8.9564E+02\n   62700.0   8.8459E+02\n   62900.0   8.7367E+02\n   63100.0   8.6287E+02\n   63300.0   8.5219E+02\n   63500.0   8.4163E+02\n   63700.0   8.3118E+02\n   63900.0   8.2084E+02\n   64200.0   8.0592E+02\n   64600.0   7.8663E+02\n   65000.0   7.6775E+02\n   65400.0   7.4928E+02\n   65800.0   7.3189E+02\n   66200.0   7.1418E+02\n   66600.0   6.9751E+02\n   67000.0   6.8186E+02\n   67400.0   6.6587E+02\n   67800.0   6.5054E+02\n   68200.0   6.3571E+02\n   68600.0   6.2125E+02\n   69000.0   6.0720E+02\n   69400.0   5.9356E+02\n   69800.0   5.8032E+02\n   70200.0   5.6740E+02\n   70600.0   5.5485E+02\n   71000.0   5.4267E+02\n   71400.0   5.3085E+02\n   71800.0   5.1931E+02\n   72200.0   5.0805E+02\n   72600.0   4.9712E+02\n   73000.0   4.8651E+02\n   73400.0   4.7616E+02\n   73800.0   4.6606E+02\n   74200.0   4.5500E+02\n   74600.0   4.3252E+02\n   75000.0   4.2328E+02\n   75400.0   4.2687E+02\n   75800.0   4.1956E+02\n   76200.0   4.1098E+02\n   76600.0   4.0261E+02\n   77000.0   3.9445E+02\n   77400.0   3.8648E+02\n   77800.0   3.7875E+02\n   78200.0   3.7116E+02\n   78600.0   3.6380E+02\n   79000.0   3.5662E+02\n   79400.0   3.4956E+02\n   79800.0   3.4273E+02\n   80200.0   3.3605E+02\n   80600.0   3.2954E+02\n   81000.0   3.2319E+02\n   81400.0   3.1699E+02\n   81800.0   3.1089E+02\n   82200.0   3.0499E+02\n   82600.0   2.9923E+02\n   83000.0   2.9357E+02\n   83400.0   2.8809E+02\n   83800.0   2.8270E+02\n   84200.0   2.7744E+02\n   84600.0   2.7231E+02\n   85000.0   2.6730E+02\n   85400.0   2.6238E+02\n   85800.0   2.5762E+02\n   86200.0   2.5293E+02\n   86600.0   2.4836E+02\n   87000.0   2.4387E+02\n   87400.0   2.3952E+02\n   87800.0   2.3524E+02\n   88200.0   2.3107E+02\n   88600.0   2.2697E+02\n   89000.0   2.2296E+02\n   89400.0   2.1906E+02\n   89800.0   2.1525E+02\n   90200.0   2.1150E+02\n   90600.0   2.0785E+02\n   91000.0   2.0425E+02\n   91400.0   2.0075E+02\n   91800.0   1.9733E+02\n   92200.0   1.9396E+02\n   92600.0   1.9068E+02\n   93000.0   1.8745E+02\n   93400.0   1.8430E+02\n   93800.0   1.8124E+02\n   94200.0   1.7821E+02\n   94600.0   1.7527E+02\n   95000.0   1.7237E+02\n   95400.0   1.6951E+02\n   95800.0   1.6676E+02\n   96200.0   1.6401E+02\n   96600.0   1.6134E+02\n   97000.0   1.5874E+02\n   97400.0   1.5617E+02\n   97800.0   1.5368E+02\n   98200.0   1.5121E+02\n   98600.0   1.4882E+02\n   99000.0   1.4646E+02\n   99400.0   1.4413E+02\n   99800.0   1.4186E+02\n  100200.0   1.3965E+02\n  200000.0   8.9713E+00\n  400000.0   5.6061E-01\n  600000.0   1.1009E-01\n  800000.0   3.4617E-02\n 1000000.0   1.4099E-02\n 1200000.0   6.7641E-03\n 1400000.0   3.6327E-03\n 1600000.0   2.1196E-03\n'}}
RESOURCE_DIR=OUT/"resources";RESOURCE_DIR.mkdir(exist_ok=True)
for name,item in EMBEDDED_ASSETS.items():
    data=item["text"].encode()
    assert hashlib.sha256(data).hexdigest()==item["sha256"]
    (RESOURCE_DIR/Path(name).name).write_bytes(data)
manifest["embedded_resources"]={k:{x:v[x] for x in ("url","sha256")} for k,v in EMBEDDED_ASSETS.items()}
write_json(OUT/"manifest.json",manifest)

## 4. Independent magnitude integrator and deterministic tests

For response $T(\lambda)$ and detector weighting $W=1$ (energy) or $W=\lambda$ (photon), the magnitude relative to a reference is

$$M=-2.5\log_{10}\frac{\int F_{10pc}(\lambda)T(\lambda)W(\lambda)\,d\lambda}{\int F_{Vega}(\lambda)T(\lambda)W(\lambda)\,d\lambda}+m_{Vega}.$$

This independent calculation partitions the wavelength axis at every spectrum and response knot, interpolates both linearly, and integrates each interval by Simpson's rule. The product is a polynomial of degree at most three, so this is exact for that piecewise-linear representation. This is independent of pPXF's sampled weighted-mean implementation. A second explicit sampled-grid calculation must reproduce native pPXF to 1e-11 mag; Simpson versus sampled-grid differences are separately required to be below 1e-4 mag (ten times smaller than the tabulated magnitude interval). Trapezoid refinements provide an additional numerical-convergence diagnostic; they are not used to tune the data.

The spectrum must cover the entire supplied response interval, including zero endpoints. No truncated R integration is silently accepted. Tests use analytic spectra, verify reference scaling and magnitude signs, and require invalid coverage to be rejected.

In [ ]:
def response_integral(wave,flux,filter_wave,response,weighting="photon",method="simpson",refine=1):
    wave,flux=clean_curve(wave,flux)
    fw,fr=clean_curve(filter_wave,response,nonnegative=True)
    if wave[0]>fw[0] or wave[-1]<fw[-1]:raise ValueError("Incomplete full response coverage")
    if weighting not in {"photon","energy"}:raise ValueError("Unknown detector weighting")
    x=np.unique(np.concatenate((fw,wave[(wave>fw[0])&(wave<fw[-1])])) )
    if refine>1:
        x=np.sort(np.concatenate([x]+[x[:-1]+(x[1:]-x[:-1])*j/refine for j in range(1,refine)]))
    def product(xx):
        out=np.interp(xx,wave,flux)*np.interp(xx,fw,fr)
        return out*xx if weighting=="photon" else out
    y=product(x);dx=np.diff(x)
    if method=="simpson":return float(np.sum(dx*(y[:-1]+4*product((x[:-1]+x[1:])/2)+y[1:])/6))
    if method=="trapezoid":return float(np.sum(dx*(y[:-1]+y[1:])/2))
    raise ValueError("Unknown integration method")

def synthetic_mag(wave,flux,fw,fr,ref_wave,ref_flux,weighting="photon",ref_mag=0.,method="simpson",refine=1):
    a=response_integral(wave,flux,fw,fr,weighting,method,refine)
    b=response_integral(ref_wave,ref_flux,fw,fr,weighting,method,refine)
    if a<=0 or b<=0:raise ValueError("Nonpositive integrated spectrum or reference")
    return -2.5*np.log10(a/b)+ref_mag

def residual_statistics(delta):
    d=np.asarray(delta,float)
    if not np.isfinite(d).all():raise ValueError("Nonfinite magnitude residual")
    median=float(np.median(d))
    return dict(n=len(d),median_delta_mag=median,p95_abs_delta_mag=float(np.percentile(abs(d),95)),
                max_abs_delta_mag=float(abs(d).max()),centered_p95_abs_mag=float(np.percentile(abs(d-median),95)),
                frac_within_table_rounding=float(np.mean(abs(d)<=TABLE_ROUNDING_HALF_MAG)),
                median_flux_ratio=float(np.median(10**(-.4*d))))

def native_grid_mag(wave,flux,fw,fr,ref_wave,ref_flux):
    def mean_on_input_grid(x,y):
        kernel=np.interp(x,fw,fr,left=0,right=0)*x*np.gradient(x)
        return np.sum(kernel*y)/np.sum(kernel)
    return -2.5*np.log10(mean_on_input_grid(wave,flux)/mean_on_input_grid(ref_wave,ref_flux))

x=np.array([1.,2.,3.]);flat=np.ones(3)
np.testing.assert_allclose(response_integral(x,x,x,flat,"energy"),4.)
np.testing.assert_allclose(response_integral(x,x,x,flat,"photon"),26./3)
np.testing.assert_allclose(synthetic_mag(x,2*flat,x,flat,x,flat),-2.5*np.log10(2))
np.testing.assert_allclose(synthetic_mag(x,flat,x,flat,x,flat,ref_mag=.03),.03)
try:response_integral(x,flat,[.5,3.], [1.,1.])
except ValueError:pass
else:raise AssertionError("Incomplete support was accepted")
print("Analytic integration, magnitude-sign and coverage tests passed")

## 5. Predefined diagnostic conventions and optional authoritative profile

The baseline reproduces the previous experiment: pPXF Johnson--Cousins response, pPXF Vega spectrum, photon weighting and Vega=0. Controlled alternatives change one choice at a time; the combined modern-MILESpy-resource calculation is labelled as such. The +0.03 R case is only a zero-point sensitivity test: the cited paper documents that value in V, not in R.

V and I are controls for common scale versus colour-dependent residuals. They are not additional science selections. No candidate is certified by matching the table most closely. If `VERIFIED_TABLE_R_PROFILE` is supplied, its exact file hashes and evidence URL are recorded and its result is considered separately from hypotheses. Certification requires documented provenance, not a fitted scale factor.

In [ ]:
modern=np.loadtxt(RESOURCE_DIR/"vega.sed")
modern_wave,modern_vega=clean_curve(modern[:,0],modern[:,1])
modern_vega=modern_vega*(3.44e-9/np.interp(5556.,modern_wave,modern_vega))
profiles=[]
for band in ("R","V","I"):
    native_name=f"REST_FRAME/Johnson-Cousins_{band}.dat"
    fw,fr=util.read_filter(native_name,native_filters)
    clean_curve(fw,fr,True)
    modern_filter=np.loadtxt(RESOURCE_DIR/f"Generic_Bessell.{band}.dat")
    mf,mr=clean_curve(modern_filter[:,0],modern_filter[:,1],True)
    base=dict(band=band,fw=fw,fr=fr,rw=native_wave,rf=native_vega,weighting="photon",
              ref_mag=0.,unit_factor=SSP_TO_LSUN_PER_ANGSTROM,lsun=LSUN_ERG_S,authoritative=False,
              native_filter_name=native_name)
    profiles.append(dict(base,name=f"ppxf_{band}_baseline",role="previous-experiment convention"))
    profiles.append(dict(base,name=f"modern_miles_resources_{band}",fw=mf,fr=mr,rw=modern_wave,rf=modern_vega,
                         weighting="energy",role="modern MILES-team resources; not certified historical-table convention"))
    if band=="R":
        profiles.append(dict(base,name="ppxf_R_energy",weighting="energy",role="weighting-only sensitivity"))
        profiles.append(dict(base,name="ppxf_R_Hayes_reference",rw=modern_wave,rf=modern_vega,role="reference-spectrum-only sensitivity"))
        profiles.append(dict(base,name="Bessell_R_ppxf_reference",fw=mf,fr=mr,role="response-only sensitivity"))
        profiles.append(dict(base,name="ppxf_R_vega_plus003",ref_mag=.03,role="hypothetical R zero-point sensitivity, not a published R value"))
if VERIFIED_TABLE_R_PROFILE is not None:
    v=VERIFIED_TABLE_R_PROFILE
    required={"filter_path","reference_path","weighting","vega_mag","reference_at_5556", "spectrum_unit_factor","lsun_erg_s","source_url","provenance_note"}
    assert required<=v.keys() and v["source_url"] and v["provenance_note"]
    fv=np.loadtxt(v["filter_path"]);rv=np.loadtxt(v["reference_path"])
    fw,fr=clean_curve(fv[:,0],fv[:,1],True);rw,rf=clean_curve(rv[:,0],rv[:,1])
    if v["reference_at_5556"] is not None:rf=rf*(float(v["reference_at_5556"])/np.interp(5556.,rw,rf))
    profiles.append(dict(name="verified_table_R",band="R",fw=fw,fr=fr,rw=rw,rf=rf,
        weighting=v["weighting"],ref_mag=float(v["vega_mag"]),unit_factor=float(v["spectrum_unit_factor"]),
        lsun=float(v["lsun_erg_s"]),authoritative=True,role=v["provenance_note"]))
    manifest["verified_resource_hashes"]={str(v[k]):sha256(v[k]) for k in ("filter_path","reference_path")}
profile_manifest=[]
for p in profiles:
    item={k:v for k,v in p.items() if k not in {"fw","fr","rw","rf"}}
    for k in ("fw","fr","rw","rf"):item[k+"_array_hash"]=hashlib.sha256(p[k].tobytes()).hexdigest()
    profile_manifest.append(item)
write_json(OUT/"profiles.json",profile_manifest)
display(pd.DataFrame(profile_manifest)[["name","band","weighting","ref_mag","authoritative","role"]])
fig,ax=plt.subplots(1,2,figsize=(11,3.8),layout="constrained")
for p in profiles:
    if p["name"] in {"ppxf_R_baseline","Bessell_R_ppxf_reference","verified_table_R"}:
        ax[0].plot(p["fw"],p["fr"]/p["fr"].max(),label=p["name"])
ax[0].set(xlabel="Wavelength [A]",ylabel="Normalized R response");ax[0].legend(fontsize=8)
ax[1].plot(native_wave,native_vega,label="pPXF Vega")
ax[1].plot(modern_wave,modern_vega,label="MILES-team Vega, Hayes-normalized")
ax[1].set(xlim=(4500,9500),xlabel="Wavelength [A]",ylabel="Vega physical F_lambda at Earth")
ax[1].legend(fontsize=8);fig.savefig(OUT/"response_and_reference.png",dpi=150);plt.show()

## 6. Direct absolute magnitudes for all selected SSPs

For operational $L_\odot$/Angstrom spectra, $F_{10pc}=S_\lambda L_\odot/[4\pi(10\,pc)^2]$. The physical solar luminosity constant here converts the spectrum's unit to cgs; it is not a solar R magnitude and does not introduce a stellar-mass array.

The direct original-grid baseline is compared with native pPXF `mag_spectrum`. R is also measured after pPXF full-spectrum log rebinning at 20.5 km/s to isolate the sampling change used in the earlier SPS helper. Every magnitude is compared with its matching table row. Failure of a numerical-accuracy assertion stops the run rather than being interpreted as physical disagreement.

All SSPs remain unscaled by their table magnitudes. A constant median residual may be displayed as a diagnostic, but it is never subtracted from the data to manufacture agreement.

In [ ]:
measurements=[];provenance=[];numerics=[]
for j,path in enumerate(files):
    age,metal=identity(path);row=phot[(age,metal)]
    with fits.open(path,memmap=True) as h:
        hdr=h[0].header
        wave=hdr["CRVAL1"]+hdr["CDELT1"]*np.arange(hdr["NAXIS1"])
        raw=np.asarray(h[0].data,float)
        clean_curve(wave,raw)
        header_units=hdr.get("BUNIT","")
        n_hdu=len(h)
    original=ORIGINAL_SSP_DIR/path.name
    original_status="missing_original_counterpart";max_data_error=None
    if original.is_file():
        with fits.open(original,memmap=True) as h:
            oh=h[0].header;ow=oh["CRVAL1"]+oh["CDELT1"]*np.arange(oh["NAXIS1"])
            if np.array_equal(wave,ow) and h[0].data.shape==raw.shape:
                max_data_error=float(np.max(abs(np.asarray(h[0].data,float)-raw)))
                original_status="identical_spectrum" if max_data_error==0 else "spectral_data_differ"
            else:original_status="wavelength_or_shape_differ"
    provenance.append(dict(file=path.name,age=age,metal=metal,bunit=header_units,
        n_hdu=n_hdu,original_status=original_status,
        max_data_error=max_data_error,safe_hash=sha256(path),
        original_hash=sha256(original) if original.is_file() else None))
    for p in profiles:
        f10=raw*p["unit_factor"]*p["lsun"]/(4*np.pi*(10*PC_CM)**2)
        mag=synthetic_mag(wave,f10,p["fw"],p["fr"],p["rw"],p["rf"],p["weighting"],p["ref_mag"])
        mag4=synthetic_mag(wave,f10,p["fw"],p["fr"],p["rw"],p["rf"],p["weighting"],p["ref_mag"],"trapezoid",4)
        mag2=synthetic_mag(wave,f10,p["fw"],p["fr"],p["rw"],p["rf"],p["weighting"],p["ref_mag"],"trapezoid",2)
        delta=mag-float(row[p["band"]])
        measurements.append(dict(file=path.name,age=age,metal=metal,profile=p["name"],band=p["band"],
            authoritative=p["authoritative"],table_mag=float(row[p["band"]]),spectral_mag=mag,
            delta_mag=delta,flux_ratio=10**(-.4*delta),trap4_minus_simpson=mag4-mag,trap4_minus_trap2=mag4-mag2))
        if p["name"].startswith("ppxf_") and p["name"].endswith("_baseline"):
            native=float(np.asarray(util.mag_spectrum(wave,f10,bands=p["native_filter_name"],system="Vega")).item())
            sampled=native_grid_mag(wave,f10,p["fw"],p["fr"],native_wave,native_vega)
            if abs(native-sampled)>KERNEL_REPRODUCTION_TOL_MAG:
                raise AssertionError(f"Native sampled-kernel reproduction failed: {path.name}")
            if abs(native-mag)>TEST_TOL_MAG:raise AssertionError(f"Native/independent magnitude integration error {path.name}: {native-mag}")
            rmag_log=None
            if p["band"]=="R":
                rebinned,logwave,_=util.log_rebin([wave[0],wave[-1]],f10,velscale=20.5)
                rmag_log=float(np.asarray(util.mag_spectrum(np.exp(logwave),rebinned,bands=p["native_filter_name"],system="Vega")).item())
            numerics.append(dict(file=path.name,age=age,metal=metal,band=p["band"],native_mag=native,
                independent_mag=mag,native_minus_independent=native-mag,native_minus_explicit_kernel=native-sampled,
                raw_vs_log_mag=None if rmag_log is None else rmag_log-native))
    if (j+1)%50==0 or j+1==len(files):print("Measured",j+1,"/",len(files),flush=True)
df=pd.DataFrame(measurements);num=pd.DataFrame(numerics);prov=pd.DataFrame(provenance)
df.to_csv(OUT/"ssp_magnitude_comparison.csv",index=False)
num.to_csv(OUT/"integration_and_sampling_checks.csv",index=False)
prov.to_csv(OUT/"safe_library_provenance.csv",index=False)
stats=[]
for name,g in df.groupby("profile",sort=False):
    stats.append(dict(profile=name,band=g.band.iloc[0],authoritative=bool(g.authoritative.iloc[0]),**residual_statistics(g.delta_mag)))
stats=pd.DataFrame(stats);stats.to_csv(OUT/"profile_summary.csv",index=False)
display(stats)
print("Maximum |native - independent| [mag]:",abs(num.native_minus_independent).max())
print("Maximum |native - explicit sampled kernel| [mag]:",abs(num.native_minus_explicit_kernel).max())
print("Maximum |raw - log-rebinned R| [mag]:",abs(num.raw_vs_log_mag.dropna()).max())
display(prov.original_status.value_counts().to_frame("SSPs"))

## 7. Plots and distinguishing a common scale from a colour-dependent residual

Plots preserve the predefined profile order. The 0.0005-mag band shows half a table rounding interval; it is a precision reference, not a guarantee that separately generated model predictions must agree that closely. The median-centered statistic measures residual shape only and is not a calibration correction. V-R and R-I residuals under the baseline cancel a wavelength-independent absolute scale and help identify colour-dependent disagreement.

In [ ]:
rprofiles=[p["name"] for p in profiles if p["band"]=="R"]
plot_rows=int(np.ceil(len(rprofiles)/3))
fig,axes=plt.subplots(plot_rows,3,figsize=(15,4*plot_rows),layout="constrained",squeeze=False)
for ax,name in zip(axes.flat,rprofiles):
    g=df[df.profile==name]
    im=ax.scatter(g.age,g.delta_mag,c=g.metal,s=12,cmap="viridis")
    ax.axhline(0,color="0.3",lw=.7);ax.axhspan(-TABLE_ROUNDING_HALF_MAG,TABLE_ROUNDING_HALF_MAG,color="0.7",alpha=.3)
    ax.set_xscale("log");ax.set(xlabel="SSP age [Gyr]",ylabel="M_spectrum - M_table [mag]",title=name)
    fig.colorbar(im,ax=ax,label="[M/H]")
for unused_ax in list(axes.flat)[len(rprofiles):]:unused_ax.set_visible(False)
fig.suptitle("R-band photometry diagnostics: hypotheses, not residual-selected calibrations")
fig.savefig(OUT/"R_residuals_by_age_metallicity.png",dpi=150);plt.show()
fig,ax=plt.subplots(1,2,figsize=(12,4),layout="constrained")
for name in rprofiles:
    g=df[df.profile==name]
    ax[0].hist(g.delta_mag,bins=min(40,max(5,len(g)//8)),histtype="step",label=name)
    ax[1].plot(np.sort(g.delta_mag),np.linspace(0,1,len(g)),label=name)
ax[0].set(xlabel="R magnitude residual [mag]",ylabel="SSPs");ax[0].legend(fontsize=7)
ax[1].set(xlabel="R magnitude residual [mag]",ylabel="Cumulative fraction")
fig.savefig(OUT/"R_residual_distributions.png",dpi=150);plt.show()
baseline=df[df.profile.isin(["ppxf_R_baseline","ppxf_V_baseline","ppxf_I_baseline"])].pivot(index=["file","age","metal"],columns="band",values="delta_mag")
baseline["delta_VminusR"]=baseline.V-baseline.R
baseline["delta_RminusI"]=baseline.R-baseline.I
baseline.to_csv(OUT/"baseline_colour_residuals.csv")
fig,ax=plt.subplots(1,2,figsize=(11,3.7),layout="constrained")
for a,column in zip(ax,["delta_VminusR","delta_RminusI"]):
    g=baseline.reset_index();a.scatter(g.age,g[column],c=g.metal,s=12,cmap="viridis")
    a.set_xscale("log");a.axhline(0,color="0.3",lw=.7)
    a.set(xlabel="SSP age [Gyr]",ylabel=column+" [mag]")
fig.savefig(OUT/"baseline_colour_residuals.png",dpi=150);plt.show()

## 8. Solar conversion and the earlier experiment: a separate layer

The primary magnitude comparison above needs no solar R magnitude. This section computes solar R magnitudes under the same explicitly specified response/reference conventions, then demonstrates that the spectral/table luminosity ratio equals `10**(-0.4*delta_mag)` regardless of the common solar magnitude used. It also compares the new original-grid baseline with the previous log-sampled SSP diagnostic, if that experiment directory is available.

This identifies a solar-conversion bookkeeping problem separately from spectral versus tabulated magnitudes. Derived masses are diagnostic reconstructions, not independently obtained SPS mass predictions. Changing a solar constant is not a permitted empirical fix merely because it reduces the residual.

In [ ]:
solar_rows=[]
for p in profiles:
    if p["band"]!="R":continue
    msun=synthetic_mag(native_wave,native_sun,p["fw"],p["fr"],p["rw"],p["rf"],p["weighting"],p["ref_mag"])
    g=df[df.profile==p["name"]].copy()
    lum_table=10**(-.4*(g.table_mag-msun));lum_spectral=10**(-.4*(g.spectral_mag-msun))
    np.testing.assert_allclose(lum_spectral/lum_table,g.flux_ratio,rtol=1e-13)
    solar_rows.append(dict(profile=p["name"],solar_R_mag=msun,
        note="pPXF solar spectrum integrated under this profile; not certified publisher solar convention"))
solar=pd.DataFrame(solar_rows);solar.to_csv(OUT/"solar_convention_diagnostic.csv",index=False);display(solar)
previous=PREVIOUS_RESULTS/"ssp_photometry_comparison.csv"
if previous.is_file():
    old=pd.read_csv(previous);old["age"]=old.age.round(2);old["metal"]=old.metal.round(2)
    now=df[df.profile=="ppxf_R_baseline"].merge(old,on=["age","metal"],validate="one_to_one")
    now["ratio_difference_from_previous"]=now.flux_ratio-now.luminosity_ratio
    now.to_csv(OUT/"previous_experiment_R_comparison.csv",index=False)
    print("Baseline luminosity-ratio max change versus previous log-sampled experiment:",abs(now.ratio_difference_from_previous).max())
else:print("Previous SSP diagnostic unavailable: independent raw-spectrum check still complete")

## 9. Optional saved-weight propagation: diagnostic, no new fits

With `RUN_WEIGHT_PROPAGATION=True` and the full SSP set selected, propagate each R candidate through the previously saved fresh weights. This predicts how the alternative photometric definition would change the earlier M/L comparison. It is labelled a **diagnostic physical-ratio calculation**, not a new native pPXF fit or an independently calibrated result. The R magnitude proxies share a common zero point, so the common solar factor cancels. No candidate is selected by its apparent agreement.

For amplitudes $a_j=w_j/n_j$, table numerator $\sum a_j\Upsilon_j10^{-0.4M_{R,j}^{table}}$ and candidate denominator $\sum a_j10^{-0.4M_{R,j}^{candidate}}$ are combined. The previous native pPXF values are retained as the measured original reference. This section reads checkpoints only and writes no production products.

In [ ]:
if RUN_WEIGHT_PROPAGATION:
    assert SSP_LIMIT is None,"Mixture propagation requires every SSP"
    rows=[]
    for folder in sorted((PREVIOUS_RESULTS/"full").iterdir()):
        if not (folder/"bin_results.csv").is_file():continue
        old=pd.read_csv(folder/"bin_results.csv").sort_values("bin_id")
        grid=pd.read_csv(folder/"template_preparation_comparison.csv")
        order=[(round(float(a),2),round(float(z),2)) for a,z in zip(grid.age,grid.metal)]
        table_proxy=np.array([10**(-.4*float(phot[k]["R"])) for k in order])
        table_ml=np.array([float(phot[k]["ML_R"]) for k in order])
        predictions={name:[] for name in rprofiles}
        with h5py.File(folder/"fit_results.hdf5","r") as h:
            assert np.all(h["state"][:]==1)
            for lo in range(0,len(old),256):
                a=h["weights"][lo:lo+256]/grid.reference_norm.to_numpy()
                numerator=a@(table_proxy*table_ml)
                for name in rprofiles:
                    g=df[df.profile==name];lookup={(round(float(r.age),2),round(float(r.metal),2)):r.spectral_mag for r in g.itertuples()}
                    denominator=a@np.array([10**(-.4*lookup[k]) for k in order])
                    predictions[name].extend(numerator/denominator)
        for name,value in predictions.items():
            difference=100*(old.ML_R_light_norm_to_r.to_numpy()/np.array(value)-1)
            rows.append(dict(case=folder.name,profile=name,n=len(value),
                median_corrected_over_candidate_percent=float(np.median(difference)),
                p99_abs_percent=float(np.percentile(abs(difference),99)),max_abs_percent=float(abs(difference).max())))
    propagated=pd.DataFrame(rows);propagated.to_csv(OUT/"saved_weight_photometric_sensitivity.csv",index=False);display(propagated)
else:print("Saved-weight propagation disabled: this run checks SSP photometry only")

## 10. Evidence-based conclusion and outputs

Numerical agreement of independent and native integration validates the calculation for the named response. It does not certify that response as the table's exact one. Identical safe/original arrays rule out changes by that particular safe-copy step for inspected counterparts; they do not independently establish all original model units.

The summary deliberately separates: numerical implementation status; physical/photometric provenance status; and measured spectral/table residuals. A lower residual under a diagnostic candidate is a lead, not proof. Historical tables and spectra may also be distinct model predictions as discussed by the authors. Only a verified profile can support the strict matched-convention test, and even then the publisher's stated accuracy is distinguished from table rounding.

Outputs: `manifest.json`, `profiles.json`, `safe_library_provenance.csv`, `ssp_magnitude_comparison.csv`, `integration_and_sampling_checks.csv`, `profile_summary.csv`, colour/solar diagnostics, comparison figures and `conclusion.json`. Optional saved-weight results are separate. Inspect baseline and verified-profile rows before making a claim about intrinsic table/library disagreement.

In [ ]:
baseline_stats=stats[stats.profile=="ppxf_R_baseline"].iloc[0].to_dict()
conclusion=dict(generated_utc=datetime.datetime.now(datetime.timezone.utc).isoformat(),versions=versions,
    n_available=len(all_files),n_selected=len(files),full_grid=SSP_LIMIT is None,
    numerical_integration_status="PASS",max_abs_native_independent_mag=float(abs(num.native_minus_independent).max()),
    max_abs_native_explicit_kernel_mag=float(abs(num.native_minus_explicit_kernel).max()),
    max_abs_raw_log_R_mag=float(abs(num.raw_vs_log_mag.dropna()).max()),
    exact_table_convention_status="DOCUMENTED_PROFILE_SUPPLIED" if VERIFIED_TABLE_R_PROFILE is not None else "UNVERIFIED",
    strict_photometric_equivalence_established=False,baseline_R=baseline_stats,
    caveat="Diagnostic agreement alone does not certify a historical table convention or absolute spectrum units; no SSP was rescaled to its table magnitude.")
if VERIFIED_TABLE_R_PROFILE is not None:
    verified=stats[stats.profile=="verified_table_R"].iloc[0].to_dict()
    conclusion["verified_R_measurements"]=verified
    conclusion["strict_rounding_level_agreement"]=bool(verified["max_abs_delta_mag"]<=TABLE_ROUNDING_HALF_MAG+TEST_TOL_MAG)
    conclusion["strict_photometric_equivalence_established"]=conclusion["strict_rounding_level_agreement"]
write_json(OUT/"conclusion.json",conclusion)
write_json(OUT/"manifest.json",manifest)
display(pd.DataFrame([baseline_stats]))
print("Numerical integration:",conclusion["numerical_integration_status"])
print("Exact historical-table convention:",conclusion["exact_table_convention_status"])
print("Strict table/spectrum equivalence established:",conclusion["strict_photometric_equivalence_established"])
print("Outputs:",OUT)

## 11. Running and preserving the earlier results

This notebook is independent of `20261031_SETONIX_MAUVE_ML_Three_Methods*.ipynb`. It does not modify `Mass.py`, shared pPXF/nGIST installations, template spectra, table files or earlier results. No network access is required: small MILES-team resource files are embedded and pinned.

Use the production container and a writable Jupyter runtime. A one-CPU, modest-memory allocated session is sufficient for SSP-only checks; the 128-worker SFH setup is unnecessary. For a first numerical smoke test set `SSP_LIMIT=3`, then restore `None` for the complete grid. Save executed output under a separate `_executed.ipynb` name so the source and measured record remain distinguishable. Optional weight propagation can be enabled after examining the profiles; it requires the completed experiment's HDF5 checkpoints but performs no spectral fit.

If the authoritative profile is still unavailable, the notebook still produces a complete, reproducible diagnostic of the earlier assumptions and precisely identifies the provenance needed to settle the strict equality question. It will not convert an unverified assumption into an asserted calibration.